In [ ]:
"""
train_nano.py — NanoModel V16 Training Script
═══════════════════════════════════════════════════════════════════════════════

Changes from V15 training script
──────────────────────────────────
Loss:
  • L1 only → L1 + perceptual (VGG relu2_2) + TV on flow
    Perceptual loss is the single highest-impact quality improvement.
    L1 trains the model to minimise pixel error; VGG perceptual loss
    trains it to preserve texture and edge structure. The difference
    is immediately visible — perceptual-trained outputs are sharp,
    L1-trained outputs are blurry.
  • TV (total variation) on flow: discourages noisy flow fields that
    cause checkerboard artifacts on uniform surfaces.

Data:
  • Crop 256×256 → 512×512 (4× more context per sample)
    Flow estimation at 256px never sees multi-object scenes;
    at 512px the model learns actual occlusion patterns.
  • Batch 16 → 8 (to fit 512×512 in 4GB VRAM)
  • Temporal augmentation: random frame-order reversal teaches
    the model that t=0.5 is symmetric (flow from A→B = -flow B→A)
  • Temporal subsampling: skip every other frame with 30% probability
    Forces the model to handle larger motion gaps.
  • Color jitter: trains robustness to lighting variation across frames.

Optimiser:
  • AdamW unchanged (good choice for this architecture)
  • Flat LR → cosine annealing with linear warmup
    Warmup prevents early divergence when perceptual loss is active.
    Cosine decay lets the model fine-tune in the last 20% of training.

Checkpointing:
  • Best validation loss checkpoint (not just latest epoch)
  • Loss breakdown logged separately for debugging
"""

import os
import sys
import platform
import time

if platform.system() == "Windows":
    os.environ["TORCHDYNAMO_DISABLE"] = "1"

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.cuda.amp import autocast, GradScaler
import torchvision.models as tvm
import pandas as pd
import numpy as np
import cv2

sys.path.append(os.path.abspath("../src"))
from nano import NanoModel

# ─────────────────────────────────────────────
#  CONFIGURATION
# ─────────────────────────────────────────────
BATCH_SIZE       = 8          # 16 OOM at 512×512 on 4GB — use 8
EPOCHS           = 80         # more epochs, cosine schedule handles LR
LEARNING_RATE    = 2e-4
WARMUP_EPOCHS    = 3          # linear warmup to prevent early divergence
NUM_WORKERS      = 0          # keep 0 on Windows (fork issues)
CROP_SIZE        = 512        # was 256 — doubles effective flow resolution
VAL_SPLIT        = 0.05       # 5% of data held out for validation
IMG_PATH         = "../dataset/raw/images"
CSV_PATH         = "../dataset/raw/log.csv"
MODEL_SAVE_PATH  = "../checkpoints/nano_v17.pth"
BEST_SAVE_PATH   = "../checkpoints/nano_v17_best.pth"

# Loss weights — tune these if one term dominates
W_L1          = 1.0
W_PERCEPTUAL  = 0.08    # higher = sharper but can over-smooth fine detail
W_TV          = 0.005   # higher = smoother flow but loses sharp motion boundaries


# ─────────────────────────────────────────────
#  PERCEPTUAL LOSS  (VGG16 relu2_2)
#  Frozen — only used to extract features, not trained.
#  relu2_2 gives a good balance: early enough to capture edges and
#  texture, late enough to ignore exact pixel positions.
# ─────────────────────────────────────────────
class PerceptualLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        vgg    = tvm.vgg16(weights=tvm.VGG16_Weights.DEFAULT).features
        # relu2_2 is layer index 9 (0-indexed)
        self.feat = nn.Sequential(*list(vgg.children())[:9]).to(device)
        for p in self.feat.parameters():
            p.requires_grad = False
        self.feat.eval()
        # VGG was trained on ImageNet with these stats
        mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
        self.register_buffer('mean', mean)
        self.register_buffer('std',  std)

    @torch.no_grad()
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """
        pred, target: (B, 3, H, W) float32 BGR [0,1]
        Returns scalar loss.
        Note: VGG expects RGB — we flip channels here.
        """
        # BGR → RGB, normalise to ImageNet stats
        p = (pred[:,   [2,1,0]] - self.mean) / self.std
        t = (target[:, [2,1,0]] - self.mean) / self.std
        return F.l1_loss(self.feat(p), self.feat(t))


# ─────────────────────────────────────────────
#  TV LOSS ON FLOW
#  Total variation encourages spatial smoothness in predicted flow.
#  Without this, the model produces pixel-level noise in uniform
#  regions which shows up as shimmering in the interpolated output.
# ─────────────────────────────────────────────
def tv_loss(flow: torch.Tensor) -> torch.Tensor:
    """flow: (B, 2, H, W)"""
    dy = (flow[:, :, 1:, :] - flow[:, :, :-1, :]).abs().mean()
    dx = (flow[:, :, :, 1:] - flow[:, :, :, :-1]).abs().mean()
    return dy + dx


# ─────────────────────────────────────────────
#  DATASET
# ─────────────────────────────────────────────
class InputDataset(Dataset):
    def __init__(self, crop_size: int = CROP_SIZE):
        if not os.path.exists(IMG_PATH):
            raise FileNotFoundError(f"Image path not found: {IMG_PATH}")

        self.crop_size = crop_size
        self.frames    = sorted(
            [f for f in os.listdir(IMG_PATH) if f.endswith(".jpg")])

        self.data_map = {}
        if os.path.exists(CSV_PATH):
            log = pd.read_csv(CSV_PATH)
            for _, row in log.iterrows():
                fid = int(row['frame_id'])
                try:
                    dx = float(row['dx']) / 50.0
                    dy = float(row['dy']) / 50.0
                    w  = float(row['w'])
                    a  = float(row['a'])
                    s  = float(row['s'])
                    d  = float(row['d'])
                    self.data_map[fid] = np.array(
                        [dx, dy, w, a, s, d], dtype=np.float32)
                except (KeyError, ValueError):
                    self.data_map[fid] = np.zeros(6, dtype=np.float32)

        print(f"  📂 Dataset: {len(self)} triplets from {len(self.frames)} frames")

    def __len__(self):
        return max(0, len(self.frames) - 2)

    def _load(self, name: str) -> np.ndarray:
        path = os.path.join(IMG_PATH, name)
        img  = cv2.imread(path)
        if img is None:
            raise IOError(f"Could not read: {path}")
        return img

    def _random_crop(self, *imgs):
        h, w = imgs[0].shape[:2]
        cs   = self.crop_size
        if h < cs or w < cs:
            # Resize up if source is smaller than crop (shouldn't happen normally)
            imgs = tuple(
                cv2.resize(im, (max(w, cs), max(h, cs))) for im in imgs)
            h, w = imgs[0].shape[:2]
        y = np.random.randint(0, h - cs)
        x = np.random.randint(0, w - cs)
        return tuple(im[y:y+cs, x:x+cs] for im in imgs)

    def __getitem__(self, idx: int):
        # ── Temporal subsampling (30% of the time) ────────────────────────
        # Skip the middle frame and use frames [0, 2, 4] instead of [0, 1, 2].
        # Forces the model to handle larger motion gaps → better quality at 4x/8x.
        if np.random.rand() < 0.3 and idx + 3 < len(self.frames):
            n0 = self.frames[idx]
            n1 = self.frames[idx + 2]   # target is 2 frames ahead
            n2 = self.frames[idx + 4] if idx + 4 < len(self.frames) else self.frames[idx + 2]
        else:
            n0 = self.frames[idx]
            n1 = self.frames[idx + 1]
            n2 = self.frames[idx + 2]

        img0   = self._load(n0)
        img1   = self._load(n2)
        target = self._load(n1)

        # ── Random crop ───────────────────────────────────────────────────
        img0, img1, target = self._random_crop(img0, img1, target)

        fid    = int(n1.split('.')[0])
        inputs = self.data_map.get(fid, np.zeros(6, dtype=np.float32))

        # ── Horizontal flip ───────────────────────────────────────────────
        if np.random.rand() > 0.5:
            img0   = cv2.flip(img0,   1)
            img1   = cv2.flip(img1,   1)
            target = cv2.flip(target, 1)
            inputs = inputs.copy()
            inputs[0]    = -inputs[0]          # flip dx
            inputs[3], inputs[5] = inputs[5], inputs[3]   # swap a/d

        # ── Temporal reversal ─────────────────────────────────────────────
        # Swap img0 and img1 with 50% probability.
        # Teaches the model that interpolation is symmetric:
        # interp(A, B, t=0.5) = interp(B, A, t=0.5)
        if np.random.rand() > 0.5:
            img0, img1 = img1, img0

        # ── Color jitter ──────────────────────────────────────────────────
        # Randomise brightness/contrast independently per frame pair.
        # Trains robustness to lighting changes between captured frames.
        if np.random.rand() > 0.5:
            alpha = np.random.uniform(0.8, 1.2)   # contrast
            beta  = np.random.randint(-20, 20)     # brightness
            img0  = np.clip(img0.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
            img1  = np.clip(img1.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

        # ── To tensor ─────────────────────────────────────────────────────
        t0 = torch.from_numpy(img0  ).permute(2, 0, 1).float().div_(255.)
        t1 = torch.from_numpy(img1  ).permute(2, 0, 1).float().div_(255.)
        tt = torch.from_numpy(target).permute(2, 0, 1).float().div_(255.)
        tv = torch.from_numpy(inputs)
        return t0, t1, tt, tv


# ─────────────────────────────────────────────
#  LR SCHEDULE: linear warmup + cosine decay
# ─────────────────────────────────────────────
def lr_lambda(epoch: int, warmup: int, total: int) -> float:
    if epoch < warmup:
        return (epoch + 1) / warmup          # linear warmup
    progress = (epoch - warmup) / max(total - warmup, 1)
    return 0.5 * (1.0 + np.cos(np.pi * progress))   # cosine decay


# ─────────────────────────────────────────────
#  TRAINING
# ─────────────────────────────────────────────
def train():
    if not torch.cuda.is_available():
        print("❌ No GPU detected."); return

    device = torch.device("cuda")
    print(f"🚀 Training NanoModel V16 on {torch.cuda.get_device_name(0)}")
    print(f"   Crop: {CROP_SIZE}×{CROP_SIZE}  |  Batch: {BATCH_SIZE}  |  Epochs: {EPOCHS}")
    print(f"   Loss weights — L1:{W_L1}  Perceptual:{W_PERCEPTUAL}  TV:{W_TV}")

    # ── Model ─────────────────────────────────────────────────────────────
    model = NanoModel().to(device)

    # Load V15 weights where possible — channel count changed so strict=False
    for path in [MODEL_SAVE_PATH, "../checkpoints/nano_v16.pth"]:
        if os.path.exists(path):
            try:
                state   = torch.load(path, map_location=device, weights_only=True)
                missing, unexpected = model.load_state_dict(state, strict=False)
                missing = [k for k in missing if k != '_t']
                loaded  = len(state) - len(missing) - len(unexpected)
                print(f"🔄 Partial load from '{path}': {loaded} keys matched")
                if missing:    print(f"   New keys (random init): {missing[:5]}…")
                if unexpected: print(f"   Dropped keys: {unexpected[:3]}…")
            except Exception as e:
                print(f"⚠️ Could not load '{path}': {e} — starting fresh.")
            break

    # ── Losses ────────────────────────────────────────────────────────────
    perceptual = PerceptualLoss(device)

    # ── Optimiser + schedule ──────────────────────────────────────────────
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda e: lr_lambda(e, WARMUP_EPOCHS, EPOCHS))
    scaler    = GradScaler()

    # ── Data ──────────────────────────────────────────────────────────────
    full_ds  = InputDataset(crop_size=CROP_SIZE)
    n_val    = max(1, int(len(full_ds) * VAL_SPLIT))
    n_train  = len(full_ds) - n_val
    train_ds, val_ds = random_split(
        full_ds, [n_train, n_val],
        generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True)

    print(f"✅ {n_train} train  |  {n_val} val samples")

    best_val_loss = float('inf')
    early_stop_counter = 0
    PATIENCE = 10

    for epoch in range(EPOCHS):
        # ── TRAIN ─────────────────────────────────────────────────────────
        model.train()
        t0            = time.perf_counter()
        total_l1      = total_perc = total_tv = total_loss = 0.0

        for i, (img0, img1, target, _) in enumerate(train_loader):
            img0   = img0.to(device,   non_blocking=True)
            img1   = img1.to(device,   non_blocking=True)
            target = target.to(device, non_blocking=True)

            imgs = torch.cat([img0, img1], dim=1)   # (B, 6, H, W)

            with autocast():
                model.set_timestamp(0.5)
                output, flow = model(imgs, return_flow=True)

                l1   = F.l1_loss(output, target)
                perc = perceptual(output, target)
                tv   = tv_loss(flow)
                loss = W_L1 * l1 + W_PERCEPTUAL * perc + W_TV * tv

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()

            # Gradient clipping — prevents occasional spikes from
            # perceptual loss when the model is far from convergence
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()

            total_l1   += l1.item()
            total_perc += perc.item()
            total_tv   += tv.item()
            total_loss += loss.item()

            if i % 20 == 0:
                lr = optimizer.param_groups[0]['lr']
                print(f"  Ep {epoch:03d} [{i:04d}/{len(train_loader)}]  "
                      f"loss={loss.item():.4f}  "
                      f"l1={l1.item():.4f}  "
                      f"perc={perc.item():.4f}  "
                      f"tv={tv.item():.5f}  "
                      f"lr={lr:.2e}",
                      end='\r')

        n       = len(train_loader)
        elapsed = time.perf_counter() - t0
        print(f"\n🟢 Ep {epoch:03d}  "
              f"loss={total_loss/n:.5f}  "
              f"l1={total_l1/n:.5f}  "
              f"perc={total_perc/n:.5f}  "
              f"tv={total_tv/n:.6f}  "
              f"({elapsed:.0f}s)")

        # ── VALIDATE ──────────────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for img0, img1, target, _ in val_loader:
                img0   = img0.to(device,   non_blocking=True)
                img1   = img1.to(device,   non_blocking=True)
                target = target.to(device, non_blocking=True)
                imgs   = torch.cat([img0, img1], dim=1)
                model.set_timestamp(0.5)
                with autocast():
                    output, flow = model(imgs, return_flow=True)
                    l   = (W_L1 * F.l1_loss(output, target)
                           + W_PERCEPTUAL * perceptual(output, target)
                           + W_TV * tv_loss(flow))
                val_loss += l.item()

        val_loss /= max(len(val_loader), 1)
        print(f"   Val loss: {val_loss:.5f}", end="")

        # Save best checkpoint separately
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), BEST_SAVE_PATH)
            print(f"  ★ NEW BEST → {BEST_SAVE_PATH}", end="")
            early_stop_counter = 0
        else:
            early_stop_counter += 1
            print(f"  [No improvement: {early_stop_counter}/{PATIENCE}]", end="")
        print()

        # Always save latest
        torch.save(model.state_dict(), MODEL_SAVE_PATH)

        scheduler.step()

        if early_stop_counter >= PATIENCE:
            print(f"\n🛑 Early stopping triggered. No improvement for {PATIENCE} epochs.")
            break

    print(f"\n✅ Training complete. Best val loss: {best_val_loss:.5f}")
    print(f"   Best weights: {BEST_SAVE_PATH}")
    print(f"   Latest weights: {MODEL_SAVE_PATH}")


if __name__ == "__main__":
    train()